## LLM Priors Assessments

### Define functions to generate descriptions and priors for synthetic datasets

In [12]:
import re
from pathlib import Path

import pandas as pd
import pyagrum as gum
from pydantic import BaseModel, Field
from typing import Annotated
from tqdm.asyncio import tqdm

from priors.llm import extract
from priors.prompt import prepare_priors


def extract_facts(string: str) -> list[dict]:
    pattern = re.compile(r"ext_((in)?dep)\((.+)\). I=([01].\d+), NA\n")
    matches = pattern.findall(string)
    facts = []
    for match in matches:
        cit_type, _, triple, score = match
        X, Y, S = triple.split(",")
        facts.append(
            {
                "cit_type": cit_type,
                "X": int(X),
                "Y": int(Y),
                "S": set() if S == "empty" else {int(var) for var in S[1:].split("y")},
                "score": float(score),
            }
        )
    return pd.DataFrame(facts).sort_values(
        by="score", ascending=False, ignore_index=True
    )

async def generate_priors(
    bn: gum.BayesNet,
    variable_descriptions: dict[str, str] | None = None,
    prior_model: str = "gemini-2.5-flash",
    parse_model: str = "gemini-2.5-flash-lite",
) -> dict:
    seed = 2025
    gum.initRandom(seed=seed)

    priors_prompt = prepare_priors(bn, descriptions=variable_descriptions)
    priors_raw = None
    while priors_raw is None:
        priors_raw = (
            (await extract(priors_prompt, None, model=prior_model)).choices[0].message.content
        )

    valid_var_pattern = (
        r"|".join(re.escape(var) for var in bn.names())
    )
    VarType = Annotated[str, Field(pattern=valid_var_pattern)]
    class VarConstraints(BaseModel):
        forbidden: set[tuple[VarType, VarType]] = set()
        required: set[tuple[VarType, VarType]] = set()

    priors = await extract(
        prompt=priors_raw, model=parse_model, pydantic_model=VarConstraints,
    )
    true_arrows = {
        (bn.variable(id1).name(), bn.variable(id2).name())
        for id1, id2 in bn.arcs()
    }

    return {
        "priors": priors.model_dump(mode="json"),
        "forbidden_Precision": len(priors.forbidden - true_arrows) / max(len(priors.forbidden), 1),
        "required_Precision": len(priors.required & true_arrows) / max(len(priors.required), 1),
    }


async def evaluate_priors(bif_paths: list[str | Path], prior_model: str, parse_model: str = None, exclude_descriptions: bool = False) -> pd.DataFrame:
    if parse_model is None:
        parse_model = prior_model
    prior_results = []
    prior_tasks = []
    for bif_path in bif_paths:
        if isinstance(bif_path, str):
            bif_path = Path(bif_path)
        bn = gum.loadBN(str(bif_path))
        variable_descriptions = {
            name: bn.variable(name).description() for name in bn.names()
        }
        if exclude_descriptions or not any(variable_descriptions.values()):
            variable_descriptions = None
        prior_results.append(
            {
                "bn": bn,
                "title": bn.propertyWithDefault("name", "no_name"),
                "filename": bif_path.stem,
                "num_nodes": bn.size(),
                "num_edges": len(bn.arcs()),
                "variable_descriptions": variable_descriptions,
            }
        )
        prior_tasks.append(
            generate_priors(
                bn=bn,
                variable_descriptions=variable_descriptions,
                prior_model=prior_model,
                parse_model=parse_model,
            )
        )
    prior_res = await tqdm.gather(*prior_tasks)
    for res_dict, prior_res in zip(prior_results, prior_res):
        res_dict.update(prior_res)
    
    return pd.DataFrame(prior_results)

### Define experiment and helper function to evaluate the precision of LLM removed edges as causal priors

In [13]:
async def run_experiments(
    dataset_paths,
    n_runs=5,
    exclude_descriptions=True,
    prior_model="gemini-2.5-flash",
    parse_model=None,
):
    """
    Run multiple experiments and return combined results.

    Usage:
        results_no_desc = await run_experiments(bnlearn_small_datasets, n_runs=5, exclude_descriptions=True)
        results_with_desc = await run_experiments(bnlearn_small_datasets, n_runs=5, exclude_descriptions=False)
    """
    all_results = []

    for run_id in range(n_runs):
        print(f"Run {run_id + 1}/{n_runs}")
        df = await evaluate_priors(
            bif_paths=dataset_paths,
            prior_model=prior_model,
            parse_model=parse_model,
            exclude_descriptions=exclude_descriptions,
        )
        df["run_id"] = run_id
        all_results.append(df)

    return pd.concat(all_results, ignore_index=True)


def get_summary(df, metrics=["Precision", "Recall", "F-beta"]):
    """
    Get summary statistics for each dataset.

    Usage:
        summary = get_summary(results_no_desc)
    """
    summary_data = []

    for dataset in df["filename"].unique():
        dataset_df = df[df["filename"] == dataset]

        for metric in metrics:
            values = dataset_df[metric].dropna()

            if len(values) > 0:
                summary_data.append(
                    {
                        "Dataset": dataset,
                        "Metric": metric,
                        "Mean": values.mean(),
                        "Std": values.std(),
                        "Min": values.min(),
                        "Max": values.max(),
                        "Runs": len(values),
                    }
                )

    return pd.DataFrame(summary_data)


def show_report(df, metrics=["Precision", "Recall", "F-beta"]):
    """
    Get summary statistics for each dataset.

    Usage:
        summary = show_report(results_no_desc)
    """
    groups = df.groupby(["filename", "with_desc"])
    meta = groups.agg(
        {
            "num_nodes": "first",
            "num_edges": "first",
        }
    )
    meta["repeats"] = groups.size()
    meta.columns = pd.MultiIndex.from_product([["meta"], meta.columns])
    index = (
        groups["num_nodes"]
        .first()
        .reset_index()
        .sort_values(["num_nodes", "filename", "with_desc"])
        .set_index(["filename", "with_desc"])
        .index
    )
    summary = pd.concat([meta, groups[metrics].agg(["mean", "std"])], axis=1)
    return summary.loc[index]

### Experiment Configurations

In [18]:
repeats = 5
prior_model = "gemini-2.5-flash"
parse_model = "gemini-2.5-flash-lite"


In [15]:
synthetic_datasets = list(Path("synthetic").glob("*.bifxml"))
res_desc = await run_experiments(
    synthetic_datasets,
    n_runs=repeats,
    prior_model=prior_model,
    parse_model=parse_model,
    exclude_descriptions=False,
)
res_desc.drop(columns=["bn"]).to_json("results_synthetic_with_desc.json", orient="records", indent=4)
print("First half done")
res_desc

Run 1/5


100%|██████████| 54/54 [01:36<00:00,  1.78s/it]


Run 2/5


100%|██████████| 54/54 [01:54<00:00,  2.13s/it] 


Run 3/5


...

100%|██████████| 54/54 [01:40<00:00,  1.85s/it]


Run 5/5


100%|██████████| 54/54 [01:39<00:00,  1.85s/it]

First half done


,bn,title,filename,num_nodes,num_edges,variable_descriptions,priors,forbidden_Precision,required_Precision,run_id
0,"BN{nodes: 5, arcs: 4, domainSize: 32, dim: 10,...",dag_5_nodes_5_edges_semantics_SF,dag_5_nodes_5_edges_semantics_SF,5,4,{'tumors': 'The presence of abnormal cellular ...,"{'forbidden': [['chronic_pain', 'tumors'], ['c...",1.000000,0.800000,0
1,"BN{nodes: 15, arcs: 14, domainSize: 32768, dim...",dag_15_nodes_15_edges_none_SF,dag_15_nodes_15_edges_none_SF,15,14,"{'chest_pain': 'Any discomfort, aching, or sha...","{'forbidden': [['heart_attacks', 'cardiovascul...",1.000000,0.187500,0
2,"BN{nodes: 5, arcs: 7, domainSize: 32, dim: 17,...",dag_5_nodes_7_edges_none_random,dag_5_nodes_7_edges_none_random,5,7,{'coronary_artery_disease': 'A specific condit...,"{'forbidden': [['ventricular_fibrillation', 'm...",1.000000,0.800000,0
3,"BN{nodes: 10, arcs: 9, domainSize: 1024, dim: ...",dag_10_nodes_10_edges_semantics_SF,dag_10_nodes_10_edges_semantics_SF,10,9,{'cardiovascular_disease': 'A broad category e...,"{'forbidden': [['blockages', 'alcoholism'], ['...",1.000000,0.275862,0
4,"BN{nodes: 15, arcs: 14, domainSize: 32768, dim...",dag_15_nodes_22_edges_semantics_SF,dag_15_nodes_22_edges_semantics_SF,15,14,{'hormonal_imbalances': 'A state characterized...,"{'forbidden': [['tensions', 'situations'], ['i...",0.888889,0.200000,0
...,...,...,...,...,...,...,...,...,...,...
265,"BN{nodes: 5, arcs: 5, domainSize: 32, dim: 11,...",dag_5_nodes_5_edges_none_random,dag_5_nodes_5_edges_none_random,5,5,{'prejudice': 'The formation of preconceived n...,"{'forbidden': [['prejudice', 'substance_abuse'...",0.857143,0.600000,4
266,"BN{nodes: 5, arcs: 7, domainSize: 32, dim: 17,...",dag_5_nodes_7_edges_none_ER,dag_5_nodes_7_edges_none_ER,5,7,{'numbness': 'A physical sensation characteriz...,"{'forbidden': [['angina', 'coronary_heart_dise...",1.000000,1.000000,4
267,"BN{nodes: 10, arcs: 9, domainSize: 1024, dim: ...",dag_10_nodes_10_edges_degrees_SF,dag_10_nodes_10_edges_degrees_SF,10,9,{'hypotension': 'A medical state defined by ab...,"{'forbidden': [['cardiac_tamponade', 'severe_v...",1.000000,0.250000,4
268,"BN{nodes: 5, arcs: 4, domainSize: 32, dim: 10,...",dag_5_nodes_7_edges_degrees_SF,dag_5_nodes_7_edges_degrees_SF,5,4,{'eye_problems': 'This variable represents the...,"{'forbidden': [['eye_problems', 'damage_to_the...",0.923077,1.000000,4


### Run experiments on bnlearn datasets with/without variable descriptions

In [16]:
bnlearn_small_datasets = list(Path("bnlearn/").glob("*.bifxml"))

In [19]:
bnlearn_results_with_desc = await run_experiments(
    bnlearn_small_datasets,
    n_runs=repeats,
    exclude_descriptions=False,
    prior_model=prior_model,
    parse_model=parse_model,
)
# Save complete results for the record
bnlearn_results_with_desc.drop(columns="bn").to_json("results_bnlearn_with_desc.json", orient="records", indent=4)

Run 1/5


100%|██████████| 5/5 [00:36<00:00,  7.34s/it]


Run 2/5


100%|██████████| 5/5 [01:02<00:00, 12.41s/it]


Run 3/5


100%|██████████| 5/5 [01:23<00:00, 16.67s/it]


Run 4/5


100%|██████████| 5/5 [00:46<00:00,  9.33s/it]


Run 5/5


100%|██████████| 5/5 [00:50<00:00, 10.10s/it]


## Priors Aggregation

We use intersection of all runs to get the consensus

In [20]:
from pathlib import Path

import pyagrum as gum
import pandas as pd

from priors.schema import Constraints


def aggregate_priors(priors_path: str, bifs_path: str):
    res = []
    df = pd.read_json(priors_path)
    for dataset in df["filename"].unique():
        bn = gum.loadBN(str(Path(bifs_path) / f"{dataset}.bifxml"))
        true_arrows = {
            (bn.variable(id1).name(), bn.variable(id2).name()) for id1, id2 in bn.arcs()
        }

        df_sub = df[df["filename"] == dataset]
        priors = [Constraints(**prior_dict) for prior_dict in df_sub["priors"]]
        majority_priors = Constraints(
            forbidden=set.intersection(*[priors.forbidden for priors in priors]),
            required=set.intersection(*[priors.required for priors in priors]),
        )

        forbidden_metrics = {
            "forbidden_length": len(majority_priors.forbidden),
            "forbidden_Precision": len(majority_priors.forbidden - true_arrows)
            / max(len(majority_priors.forbidden), 1),
            "forbidden_Recall": len(majority_priors.forbidden - true_arrows)
            / (bn.size() * (bn.size() - 1) - len(true_arrows)),
        }
        forbidden_metrics["forbidden_F1"] = (
            2
            * forbidden_metrics["forbidden_Precision"]
            * forbidden_metrics["forbidden_Recall"]
            / max(
                forbidden_metrics["forbidden_Precision"]
                + forbidden_metrics["forbidden_Recall"],
                1e-6,
            )
        )

        required_metics = {
            "required_length": len(majority_priors.required),
            "required_Precision": len(majority_priors.required & true_arrows)
            / max(len(majority_priors.required), 1),
            "required_Recall": len(majority_priors.required & true_arrows)
            / (len(true_arrows)),
        }
        required_metics["required_F1"] = (
            2
            * required_metics["required_Precision"]
            * required_metics["required_Recall"]
            / max(
                required_metics["required_Precision"]
                + required_metics["required_Recall"],
                1e-6,
            )
        )
        res.append(
            {
                **df_sub[
                    ["filename", "num_nodes", "num_edges", "variable_descriptions"]
                ]
                .iloc[0]
                .to_dict(),
                "priors": majority_priors.model_dump(mode="json"),
                **forbidden_metrics,
                **required_metics,
            }
        )

    return pd.DataFrame(res)

In [21]:
bnlearn_priors_aggregated = aggregate_priors(
    "results_bnlearn_with_desc.json", "bnlearn"
)
bnlearn_priors_aggregated.to_json("bnlearn.json", orient="records", indent=4)
bnlearn_priors_aggregated

,filename,num_nodes,num_edges,variable_descriptions,priors,forbidden_length,forbidden_Precision,forbidden_Recall,forbidden_F1,required_length,required_Precision,required_Recall,required_F1
0,survey,6,6,"{'A': 'Age: the age, recorded as young (young)...","{'forbidden': [['E', 'S'], ['R', 'A'], ['T', '...",10,1.0,0.416667,0.588235,1,1.000000,0.166667,0.285714
1,cancer,5,4,{'Dyspnoea': 'Dyspnoea: Indicates whether the ...,"{'forbidden': [['Dyspnoea', 'Pollution'], ['Xr...",8,1.0,0.500000,0.666667,4,1.000000,1.000000,1.000000
2,sachs,11,17,"{'PIP3': 'PIP3 (Phosphatidylinositol (3,4,5)-t...","{'forbidden': [['PIP3', 'PIP2'], ['Erk', 'Mek'...",5,0.8,0.043011,0.081633,5,0.400000,0.117647,0.181818
3,asia,8,8,{'dysp': 'dyspnoea: whether or not the patient...,"{'forbidden': [['bronc', 'smoke'], ['dysp', 't...",12,1.0,0.250000,0.400000,9,0.555556,0.625000,0.588235
4,earthquake,5,4,{'Burglary': 'An event involving an unauthoriz...,"{'forbidden': [['Earthquake', 'Burglary'], ['J...",10,1.0,0.625000,0.769231,0,0.000000,0.000000,0.000000


In [22]:
synthetic_priors_aggregated = aggregate_priors(
    "results_synthetic_with_desc.json", "synthetic"
)
synthetic_priors_aggregated.to_json("synthetic.json", orient="records", indent=4)
synthetic_priors_aggregated

,filename,num_nodes,num_edges,variable_descriptions,priors,forbidden_length,forbidden_Precision,forbidden_Recall,forbidden_F1,required_length,required_Precision,required_Recall,required_F1
0,dag_5_nodes_5_edges_semantics_SF,5,4,{'tumors': 'The presence of abnormal cellular ...,"{'forbidden': [['chronic_pain', 'tumors'], ['s...",4,1.000000,0.250000,0.400000,4,0.750000,0.750000,0.750000
1,dag_15_nodes_15_edges_none_SF,15,14,"{'chest_pain': 'Any discomfort, aching, or sha...","{'forbidden': [['magnesium_deficiency', 'pollu...",3,1.000000,0.015306,0.030151,6,0.166667,0.071429,0.100000
2,dag_5_nodes_7_edges_none_random,5,7,{'coronary_artery_disease': 'A specific condit...,"{'forbidden': [['ventricular_fibrillation', 'm...",3,1.000000,0.230769,0.375000,4,0.750000,0.428571,0.545455
3,dag_10_nodes_10_edges_semantics_SF,10,9,{'cardiovascular_disease': 'A broad category e...,"{'forbidden': [['premature_birth', 'alcoholism...",3,1.000000,0.037037,0.071429,5,0.400000,0.222222,0.285714
4,dag_15_nodes_22_edges_semantics_SF,15,14,{'hormonal_imbalances': 'A state characterized...,"{'forbidden': [], 'required': [['phthalates', ...",0,0.000000,0.000000,0.000000,1,1.000000,0.071429,0.133333
5,dag_10_nodes_15_edges_semantics_random,10,15,{'blockage': 'An impediment or obstruction to ...,"{'forbidden': [['abdominal_pain', 'obstruction...",2,1.000000,0.026667,0.051948,3,1.000000,0.200000,0.333333
6,dag_15_nodes_22_edges_none_SF,15,14,{'alkaloids': 'Naturally occurring organic com...,"{'forbidden': [['kidney_failure', 'environment...",8,1.000000,0.040816,0.078431,7,0.142857,0.071429,0.095238
7,dag_5_nodes_7_edges_semantics_ER,5,7,{'high_blood_pressure': 'A medical condition w...,"{'forbidden': [['sudden_death', 'heart_problem...",6,1.000000,0.461538,0.631579,6,0.833333,0.714286,0.769231
8,dag_10_nodes_15_edges_semantics_ER,10,15,"{'old_age': 'The advanced period of life, typi...","{'forbidden': [['heart_problems', 'old_age'], ...",14,1.000000,0.186667,0.314607,14,0.428571,0.400000,0.413793
9,dag_10_nodes_15_edges_degrees_ER,10,15,{'hypothyroidism': 'An endocrine disorder wher...,"{'forbidden': [['sudden_death', 'chronic_disea...",2,1.000000,0.026667,0.051948,4,0.750000,0.200000,0.315789
